# Empirical Nonlinear Observability Analysis: TB Transmission Model

This notebook analyzes the observability of the TB transmission model with the following dynamics:

$$\frac{dS}{dt} = -\rho\beta\frac{IS}{N} + \kappa R$$

$$\frac{dE}{dt} = \rho\beta\frac{IS}{N} - \alpha E$$

$$\frac{dI}{dt} = \alpha E - (\gamma + \mu)I$$

$$\frac{dR}{dt} = \gamma I - \kappa R$$

$$\frac{dD}{dt} = \mu I$$

Where:
- $S$: Susceptible
- $E$: Exposed (latent TB)
- $I$: Infected (active TB)
- $R$: Recovered
- $D$: Dead (cumulative deaths)
- $\rho$: contact rate modifier
- $\beta$: transmission rate
- $N$: total population
- $\kappa$: rate of immunity loss
- $\alpha$: rate of progression to active TB
- $\gamma$: recovery rate
- $\mu$: disease-induced mortality rate

In [ ]:
import numpy as np
import sympy as sp
from IPython.display import display
import matplotlib.pyplot as plt

In [ ]:
# Import symbolic derivatives utility
import requests
url = 'https://raw.githubusercontent.com/florisvb/Nonlinear_and_Data_Driven_Estimation/main/Utility/symbolic_derivatives.py'
r = requests.get(url)

with open('symbolic_derivatives.py', 'w') as f:
    f.write(r.text)

import symbolic_derivatives

## Define System States and Parameters

In [ ]:
# Define states
S, E, I, R, D = sp.symbols(['S', 'E', 'I', 'R', 'D'])
x = [S, E, I, R, D]

# Parameters
rho, beta, N_pop, kappa, alpha, gamma, mu = sp.symbols(['rho', 'beta', 'N', 'kappa', 'alpha', 'gamma', 'mu'])

print("States:", x)
print("Parameters: ρ, β, N, κ, α, γ, μ")

## Define System Dynamics in Control-Affine Form

We write the system as: $\dot{x} = f_0(x) + u_1 f_1(x)$

where $u_1 = \kappa$ (rate of immunity loss, which can be viewed as a control input)

In [ ]:
# f_0: drift dynamics (when κ=0)
f_0 = sp.Matrix([
    -rho*beta*I*S/N_pop,           # dS/dt when κ=0
    rho*beta*I*S/N_pop - alpha*E,  # dE/dt
    alpha*E - (gamma + mu)*I,       # dI/dt
    gamma*I,                        # dR/dt when κ=0
    mu*I                            # dD/dt
])

# f_1: control vector for κR term
f_1 = sp.Matrix([
    R,      # κR appears in dS/dt
    0,      # no direct effect on E
    0,      # no direct effect on I
    -R,     # -κR appears in dR/dt
    0       # no direct effect on D
])

print("Drift dynamics f_0:")
display(f_0)
print("\nControl vector f_1:")
display(f_1)

## Define Measurement Function

### Scenario 1: Measure Active Infections (I) and Cumulative Deaths (D)

This is the most realistic scenario - we can count active TB cases and total deaths.

In [ ]:
# Measurement function: y = h(x)
# We measure infected individuals and cumulative deaths
h = sp.Matrix([I, D])

print("Measurement function h(x):")
display(h)
print("\nWe measure:")
print("- y₁ = I (active TB cases)")
print("- y₂ = D (cumulative TB deaths)")

## Calculate Lie Derivatives

We calculate:
- $L_{f_0}h$: derivative of $h$ along $f_0$
- $L_{f_1}h$: derivative of $h$ along $f_1$

In [ ]:
# Calculate Lie derivatives
L_f0_h = symbolic_derivatives.directional_derivative(h, x, f_0)
print("L_f0(h):")
display(L_f0_h)

print("\n" + "="*60 + "\n")

L_f1_h = symbolic_derivatives.directional_derivative(h, x, f_1)
print("L_f1(h):")
display(L_f1_h)

## Construct Observability Matrix G

$$G = \begin{bmatrix} h \\ L_{f_0}h \\ L_{f_1}h \end{bmatrix}$$

In [ ]:
# Assemble observability matrix G
G = sp.Matrix([h, L_f0_h, L_f1_h])

print("Observability matrix G:")
display(G)
print(f"\nG has {len(G)} rows (measurements + derivatives)")
print(f"System has {len(x)} states")

## Calculate Jacobian of G

The system is locally observable if the Jacobian $\frac{\partial G}{\partial x}$ has full rank (equal to number of states).

In [ ]:
# Jacobian of G with respect to states
J_G = G.jacobian(x)

print("Jacobian of G with respect to states:")
display(J_G)
print(f"\nJacobian shape: {J_G.shape}")

## Check Observability at Operating Point

Using realistic TB parameters for Nigeria or similar endemic settings.

In [ ]:
# Operating point with realistic TB parameters
x0 = {
    S: 150000000,       # Susceptible population
    E: 50000000,        # Exposed (latent TB) - ~23% of population
    I: 500000,          # Active TB cases
    R: 10000000,        # Recovered individuals
    D: 2500000,         # Cumulative deaths over time
    rho: 0.8,           # Contact rate modifier
    beta: 0.000005,     # Transmission rate (calibrated)
    N_pop: 223000000,   # Total population (Nigeria)
    kappa: 0.05,        # Rate of immunity loss (slow)
    alpha: 0.1,         # Progression rate to active TB (~10 years latency)
    gamma: 0.3,         # Recovery rate (~3.3 years treatment/recovery)
    mu: 0.15            # TB mortality rate (15% of active cases)
}

print("Operating Point (Realistic TB Endemic Setting):")
print("="*60)
for var, val in x0.items():
    print(f"{var}: {val:,.0f}" if val > 1000 else f"{var}: {val}")

In [ ]:
# Evaluate Jacobian at operating point
J_G_eval = J_G.subs(x0)

print("Jacobian evaluated at operating point:")
display(J_G_eval)

# Calculate rank
rank_G = J_G_eval.rank()
n_states = len(x)

print(f"\n{'='*60}")
print(f"Rank of Jacobian: {rank_G}")
print(f"Number of states: {n_states}")
print(f"{'='*60}\n")

if rank_G == n_states:
    print("✓ SYSTEM IS LOCALLY OBSERVABLE at this operating point!")
    print("  All states can be estimated from measurements I and D.")
else:
    print("✗ SYSTEM IS NOT FULLY OBSERVABLE at this operating point.")
    print(f"  Only {rank_G} out of {n_states} states are observable.")
    print("  Additional measurements or derivatives may be needed.")

## Higher Order Derivatives (if needed)

If the system is not observable with first derivatives, we calculate second derivatives.

In [ ]:
# Calculate second-order Lie derivatives
L_f0_L_f0_h = symbolic_derivatives.directional_derivative(L_f0_h, x, f_0)
L_f0_L_f1_h = symbolic_derivatives.directional_derivative(L_f1_h, x, f_0)
L_f1_L_f0_h = symbolic_derivatives.directional_derivative(L_f0_h, x, f_1)
L_f1_L_f1_h = symbolic_derivatives.directional_derivative(L_f1_h, x, f_1)

# Extended observability matrix with second derivatives
G_extended = sp.Matrix([
    h, 
    L_f0_h, 
    L_f1_h,
    L_f0_L_f0_h,
    L_f0_L_f1_h,
    L_f1_L_f0_h,
    L_f1_L_f1_h
])

print("Extended observability matrix G (with 2nd derivatives):")
print(f"Shape: {G_extended.shape}")

# Jacobian of extended G
J_G_ext = G_extended.jacobian(x)
J_G_ext_eval = J_G_ext.subs(x0)
rank_G_ext = J_G_ext_eval.rank()

print(f"\nRank with 2nd derivatives: {rank_G_ext}/{n_states}")

if rank_G_ext == n_states:
    print("✓ System becomes observable with second derivatives!")
elif rank_G_ext > rank_G:
    print(f"  Rank improved from {rank_G} to {rank_G_ext} with 2nd derivatives.")
else:
    print("  Second derivatives did not improve observability.")

## Alternative Measurement Scenario 2: Measure I, D, and R

What if we can also measure recovered individuals (through surveys or testing)?

In [ ]:
# Alternative measurement: I, D, and R
h_alt = sp.Matrix([I, D, R])

print("Alternative measurement function:")
display(h_alt)

# Calculate Lie derivatives for alternative measurement
L_f0_h_alt = symbolic_derivatives.directional_derivative(h_alt, x, f_0)
L_f1_h_alt = symbolic_derivatives.directional_derivative(h_alt, x, f_1)

G_alt = sp.Matrix([h_alt, L_f0_h_alt, L_f1_h_alt])
J_G_alt = G_alt.jacobian(x)
J_G_alt_eval = J_G_alt.subs(x0)
rank_alt = J_G_alt_eval.rank()

print(f"\nWith measurements [I, D, R]:")
print(f"Rank: {rank_alt}/{n_states}")

if rank_alt == n_states:
    print("✓ System IS observable with these measurements!")
else:
    print(f"✗ System NOT fully observable (rank={rank_alt}/{n_states})")

## Summary and Interpretation

### Key Findings:

1. **Measurement Requirements**: 
   - With only I and D measured, we analyze if all states (S, E, I, R, D) are observable
   - Additional measurements (like R) may improve observability

2. **Practical Implications**:
   - If observable: We can estimate latent TB (E) and susceptible (S) populations from active case and death data
   - If not fully observable: Need additional data or measurements

3. **Control Considerations**:
   - The control input κ (immunity loss rate) affects observability
   - In practice, κ may be small, affecting which states can be estimated

## Visualization: Nullspace Analysis

In [ ]:
# Convert to numpy for numerical analysis
J_G_numeric = np.array(J_G_eval.tolist()).astype(float)

# Compute SVD
U, s, Vt = np.linalg.svd(J_G_numeric)

print("Singular values of Jacobian:")
for i, sv in enumerate(s):
    print(f"  σ_{i+1} = {sv:.2e}")

# Plot singular values
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.semilogy(range(1, len(s)+1), s, 'o-', linewidth=2, markersize=8)
plt.grid(True, alpha=0.3)
plt.xlabel('Index', fontsize=12)
plt.ylabel('Singular Value', fontsize=12)
plt.title('Singular Values of Observability Jacobian', fontsize=12)
plt.xticks(range(1, len(s)+1))

plt.subplot(1, 2, 2)
plt.bar(range(1, len(s)+1), s)
plt.xlabel('Index', fontsize=12)
plt.ylabel('Singular Value', fontsize=12)
plt.title('Singular Value Spectrum', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3, axis='y')
plt.xticks(range(1, len(s)+1))

plt.tight_layout()
plt.savefig('tb_observability_singular_values.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print(f"- Condition number: {s[0]/s[-1]:.2e}")
if s[-1] < 1e-10:
    print("- WARNING: Near-singular matrix - some states may be unobservable")
else:
    print("- Good conditioning - states are well-observable")

## Exercise Questions:

1. **Is the system observable with no control** (κ = 0)?
   - Evaluate the rank when κ = 0
   
2. **Is the system observable with control** (κ ≠ 0)?
   - Compare ranks with different κ values
   
3. **How many derivatives are needed**: 1st order or 2nd order?
   - Compare ranks of G vs G_extended
   
4. **What measurements are necessary**?
   - Which combinations of {I, D, R, E} give full observability?

In [ ]:
# Exercise 1: Check observability without control (κ=0)
x0_no_control = x0.copy()
x0_no_control[kappa] = 0

J_G_no_control = J_G.subs(x0_no_control)
rank_no_control = J_G_no_control.rank()

print("Exercise 1: Observability WITHOUT control (κ=0)")
print("="*60)
print(f"Rank: {rank_no_control}/{n_states}")
if rank_no_control == n_states:
    print("✓ Observable without control")
else:
    print(f"✗ NOT fully observable (missing {n_states - rank_no_control} dimensions)")

print("\n" + "="*60 + "\n")

# Exercise 2: Check with varying control values
print("Exercise 2: Observability WITH different control values")
print("="*60)
kappa_values = [0.01, 0.05, 0.1, 0.2]
for kappa_val in kappa_values:
    x0_test = x0.copy()
    x0_test[kappa] = kappa_val
    J_test = J_G.subs(x0_test)
    rank_test = J_test.rank()
    print(f"κ = {kappa_val:4.2f}: Rank = {rank_test}/{n_states} {'✓' if rank_test == n_states else '✗'}")

print("\n" + "="*60 + "\n")

# Exercise 3: Compare 1st vs 2nd order derivatives
print("Exercise 3: Derivative order requirements")
print("="*60)
print(f"With 1st derivatives only: Rank = {rank_G}/{n_states}")
print(f"With 2nd derivatives: Rank = {rank_G_ext}/{n_states}")
if rank_G == n_states:
    print("→ First derivatives are SUFFICIENT for full observability")
elif rank_G_ext == n_states:
    print("→ Second derivatives are REQUIRED for full observability")
else:
    print("→ Even 2nd derivatives insufficient - need more measurements")